# ESS BEER simulation

Example of using the `ess_beer_model` package to generate simulation files and run simulations with McStas and SIMRES.

**Repository**: `https://github.com/saroun/ess_beer_model.git`

**Installation**: from the repository root, type `pip install -e .`


In [ ]:
import beer
import beer.mcstas as mcstas
import numpy as np
deg = np.pi/180

## McStas instrument file
Generate McStas instrument file for primary beam line (source to sample slit). The generated `BEER_reference.instr` file can be used to run simulations of the reference operation modes as specified in the *BEER optics report* (ESS-0238217).

The following code prepares workspace for simulations, verifies McStas installation and creates the instrument file (primary beam line, soure to sample).

If McStas verification fails, check if mcstas is accessible in this environment. You can also compile the instrument manually using `mcgui` and then continue with next steps (running simulations). 


In [ ]:
# configure workspace and McStas environment 

mcstas.configure(workpath='./mcstas')

# optionally, set also guide waviness and misalignment 
waviness = 0.2 # RMS in mrad - equivalent with SIMRES definition
misalign=[0.02, 0.02] # spatial misfit in mm (horizontal,vertical)

# create instrument file 
# NOTE: McStas defines wavy as FWHM in deg
mcstas.create_instrument(wavy=np.sqrt(8*np.log(2))*waviness*1e-3/deg)


## Compile the instrument file

Set froce=true to recompile the instrument.  
Set mpi<>0 to compile with MPI for parallel computing.  


In [ ]:
mcstas.compile_instrument(force=True, mpi=1)

## Run simulation for selected modes

`modes`:  Give a comma-separated list of mode names  
`mpi`:  Set number of cores to use
`n`:  Number of generated neutrons  
`plot`: True to plot results.  

Note that results are automatically saved in subdirectories named by the given mode ID. 

In [ ]:
# define common arguments for execute:
param = {
'mpi':5, 
'n':1e7,
'plot':True,
'verbose':0, 
'misfit_h':misalign[0]*1e-3, 
'misfit_v':misalign[1]*1e-3
}

result = mcstas.execute(modes='F0', **param)

In [ ]:
# Run this if you need to replot the results later
# mcstas.plot_result(result, title='My simulation', pdf='myresult')

## Run selected modes

In [ ]:
# result = mcstas.execute(modes='F1, F2, PS2', **param)

## Run simulations for all modes

In [ ]:
param['n'] = 1e8
result = mcstas.execute(modes='all', **param)

# Simulation with SIMRES

Assuming that SIMRES is installed (see https://github.com/saroun/simres), run the `configure()` function below. As with McStas, specify the workspace directory. Optionally, you can also set `java` and `simresdir` directories to define the 
JRE interpreter and SIMRES installation directory, respectively.

**NOTE**: The input number if neutrons in SIMRES means the number of completed rays ariving to the last component. It should therefore be smaller than in McStas (typically 1e4 .. 1e5). Setting `n` too large may lead to a long execution time for some modes where variance reduction is not efficient (like the modulation modes). 


In [ ]:
import beer.simres as simres

simres.configure(workpath='./simres')

# set also guide waviness and misalignment 
waviness=0.2 # RMS in mrad
misalign=[0.02, 0.02] # spatial misfit in mm (horizontal,vertical)


## Update instrument setup

Create script `BEER_setup.inp`  for updating instrument configuration. Set `scriptonly=False` to also start SIMRES and run the script so that the configuration matches the instrument geometry.

In [ ]:
simres.update(scriptonly=False)

## Run simulation for all modes

Similar as `mcstas.execute()`:

`modes`:  Give a comma-separated list of mode names  
`n`:  Number of generated neutrons  
`plot`: True to plot results.  
`waviness`: Guide waviness RMS \[mrad\]  
`misalign`: Guide misalignment \[mm\]  (horizontal, vertical)

Results are saved in the output directory with the mode ID used as a prefix to all files. 

Use comma-separated mode ID's to run simulations for multiple modes. In such a case, the graphical output is simplified to spectra at the sample position for each mode. 

To reproduce the results later, you can use corresponding input scripts generated in the workspace directory (to be executed together with the `BEER_reference.xml` instrument file). 

In [ ]:
# define common arguments for execute:
param_SIMRES = {
'n':1e5,
'plot':True,
'waviness':waviness, 
'misalign':misalign
}
result = simres.execute(modes='all', **param_SIMRES)

## Repeat **McStas** for M-modes with higher statistics

In [ ]:
param = {
'mpi':5, 
'n':1e9,
'plot':'M1-M4',
'verbose':0, 
'misfit_h':misalign[0]*1e-3, 
'misfit_v':misalign[1]*1e-3
}
result = mcstas.execute(modes='M1,M2,M3,M4', **param)